<a href="https://colab.research.google.com/github/weverton/senacia-pi-iii/blob/main/Final_PI_Senac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install insightface
!pip install onnxruntime
!pip install opencv-python
!pip install scikit-learn
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.2/762.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 24.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import cv2
import pickle
import numpy as np

from insightface.app import FaceAnalysis

TRAIN_PATH = "/content/drive/MyDrive/PI Senac final"
OUTPUT_FILE = "/content/embeddings.pkl"

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CPUExecutionProvider"]
)

app.prepare(ctx_id=0)

database = {}

for person in os.listdir(TRAIN_PATH):

    person_path = os.path.join(
        TRAIN_PATH,
        person
    )

    if not os.path.isdir(person_path):
        continue

    embeddings = []

    print(f"Processando {person}")

    for image_name in os.listdir(person_path):

        image_path = os.path.join(
            person_path,
            image_name
        )

        image = cv2.imread(image_path)

        if image is None:
            continue

        faces = app.get(image)

        if len(faces) == 0:
            continue

        embeddings.append(
            faces[0].embedding
        )

    if len(embeddings) == 0:
        continue

    mean_embedding = np.mean(
        embeddings,
        axis=0
    )

    database[person] = mean_embedding

with open(
    OUTPUT_FILE,
    "wb"
) as f:

    pickle.dump(database, f)

print(
    f"Banco criado com {len(database)} pessoas."
)

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: [(128, 128), (640, 640)

In [ ]:
import os
import cv2
import pickle

from insightface.app import FaceAnalysis
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

THRESHOLD = 0.60

TEST_PATH = "/content/drive/MyDrive/Teste PI"

with open(
    "/content/embeddings.pkl",
    "rb"
) as f:

    database = pickle.load(f)

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CPUExecutionProvider"]
)

app.prepare(ctx_id=0)

y_true = []
y_pred = []

for person in os.listdir(TEST_PATH):

    person_path = os.path.join(
        TEST_PATH,
        person
    )

    if not os.path.isdir(person_path):
        continue

    for image_name in os.listdir(person_path):

        image_path = os.path.join(
            person_path,
            image_name
        )

        image = cv2.imread(image_path)

        if image is None:
            continue

        faces = app.get(image)

        if len(faces) == 0:
            continue

        embedding = faces[0].embedding

        best_name = None
        best_score = -1

        for name, db_embedding in database.items():

            score = cosine_similarity(
                embedding.reshape(1, -1),
                db_embedding.reshape(1, -1)
            )[0][0]

            if score > best_score:

                best_score = score
                best_name = name

        if best_score < THRESHOLD:
            best_name = "Desconhecido"

        y_true.append(person.strip())
        y_pred.append(best_name.strip())

        print(
            f"Real: {person:20} "
            f"Previsto: {best_name:20} "
            f"Score: {best_score:.3f}"
        )

if len(y_true) > 0:
    accuracy = sum(
        t == p
        for t, p in zip(y_true, y_pred)
    ) / len(y_true)

    print()
    print("=" * 50)
    print(f"Accuracy: {accuracy:.4f}")
    print("=" * 50)

    print()

    print(
        classification_report(
            y_true,
            y_pred,
            zero_division=0
        )
    )

    print()

    print(
        confusion_matrix(
            y_true,
            y_pred
        )
    )
else:
    print("No test data was processed or no faces were detected in the test images.")

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: [(128, 128), (640, 640)

In [ ]:
print("y_true:")
for x in y_true:
    print(repr(x))

print("\ny_pred:")
for x in y_pred:
    print(repr(x))

y_true:
'Mateus'
'Mateus'
'Mateus'
'Mateus'
'Mateus'
'JP'
'JP'
'JP'
'JP'

y_pred:
'Mateus'
'Mateus'
'Mateus'
'Mateus'
'Mateus'
'JP'
'JP'
'JP'
'JP'
